# Market Foundation Model — Kaggle GPU

Notebook resmi untuk menjalankan pipeline Market Foundation Model pada Kaggle GPU (T4/P100 atau GPU yang tersedia). Aktifkan **Internet** dan **GPU** pada Notebook Settings sebelum menjalankan.

**Protokol wajib:** tokenizer/pretraining/fine-tuning 2015–2020, pemilihan checkpoint 2021, kalibrasi 2022, final test 2023. Jangan mengubah split setelah melihat hasil final test.

Kaggle menggunakan disk kerja `/kaggle/working`. Output notebook dapat disimpan melalui **Save Version → Save & Run All**. Untuk melanjutkan training di sesi baru, tambahkan output sesi sebelumnya sebagai Kaggle Dataset lalu isi `INPUT_DATASET_DIR` di sel konfigurasi.

In [ ]:
from pathlib import Path
import os, subprocess, sys, shutil

REPO = Path('/kaggle/working/market-foundation-model')
GIT_REF = 'feature/stage3-kaggle-ddp-2gpu'  # branch with verified DDP implementation; switch to main after merge
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', GIT_REF, 'https://github.com/NodePro-gif/market-foundation-model.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', GIT_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', GIT_REF, f'origin/{GIT_REF}'], check=True)
%cd /kaggle/working/market-foundation-model
print('Repository:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())

## 1. Periksa GPU dan dependensi

Kaggle: buka **Settings → Accelerator → GPU** dan aktifkan **Internet**. Jangan install ulang PyTorch karena Kaggle biasanya sudah menyediakan build CUDA yang sesuai.

In [ ]:
!pip -q install "numpy>=1.24,<3" "pandas>=2,<4" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4"

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU belum aktif. Aktifkan Settings → Accelerator → GPU.')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB')
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('AMP: FP16 | TF32:', torch.backends.cuda.matmul.allow_tf32)

## 2. Workspace, restore dataset, dan checkpoint

Semua proses training berjalan di disk lokal `/kaggle/working`, bukan direktori input yang read-only. Jika sudah punya output sesi Kaggle sebelumnya yang diunggah sebagai Dataset, isi `INPUT_DATASET_DIR` dengan path-nya (contoh: `/kaggle/input/mfm-previous-run`). Folder `corpus`, `npz`, `mkt`, `ckpt`, dan `ftune` akan dipulihkan otomatis.

In [ ]:
from pathlib import Path
import os, shutil

WORK = Path('/kaggle/working/mfm_run')
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = Path('/kaggle/working/mfm_output')
OUTPUT.mkdir(parents=True, exist_ok=True)

# Opsional: path dataset Kaggle lama yang berisi folder corpus/ npz/ mkt/ ckpt/ ftune/.
INPUT_DATASET_DIR = ""  # contoh: "/kaggle/input/mfm-previous-run"
INPUT = Path(INPUT_DATASET_DIR) if INPUT_DATASET_DIR else None

FOLDERS = ['corpus', 'npz', 'mkt', 'ckpt', 'ftune']
for name in FOLDERS:
    dst = WORK / name
    dst.mkdir(parents=True, exist_ok=True)
    if INPUT and (INPUT / name).is_dir():
        shutil.copytree(INPUT / name, dst, dirs_exist_ok=True)
    # Restore artifacts from an earlier run saved in this notebook output, if present.
    saved = OUTPUT / name
    if saved.is_dir():
        shutil.copytree(saved, dst, dirs_exist_ok=True)

def sync_output(name):
    src, dst = WORK / name, OUTPUT / name
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Saved {name} -> {dst}')

os.environ['MFM_DATA_ROOT'] = str(WORK)
os.environ['MFM_CORPUS_DIR'] = str(WORK / 'corpus')
os.environ['MFM_NPZ_DIR'] = str(WORK / 'npz')
os.environ['MFM_MKT_DIR'] = str(WORK / 'mkt')
os.environ['MFM_CKPT_DIR'] = str(WORK / 'ckpt')
os.environ['MFM_CKPT_BACKUP_DIR'] = str(OUTPUT / 'ckpt')
os.environ['MFM_FT_BACKUP_DIR'] = str(OUTPUT / 'ftune')
os.environ['MFM_FT_DIR'] = str(WORK / 'ftune')
print('Working directory:', WORK)
print('Output directory:', OUTPUT)
print('Previous dataset:', INPUT if INPUT else 'not attached')

## 3. Unduh dan verifikasi data

Internet harus aktif. Stage 1 mengambil OHLCV dari provider yang dikonfigurasi di `stage1_corpus.py`. File yang tidak tersedia dilaporkan; XAUUSD M5 harus lengkap untuk seluruh tahun 2015–2023. Jika folder corpus sudah dipulihkan, proses akan melewati file yang sudah ada.

In [ ]:
%cd /kaggle/working/market-foundation-model
!python -u stage1_corpus.py
sync_output('corpus')

## 4. Tokenizer dan label

Tahap CPU/RAM-intensive. Statistik kuantil, regime, dan BPE hanya dipelajari dari data training 2015–2020 dan aset non-held-out.

In [ ]:
!python -u test_tokenizer.py
!python -u stage2_tokenizer.py
!ls -lh "$MFM_MKT_DIR" | head -30
sync_output('npz')
sync_output('mkt')

## 5. GPT pretraining — 2× GPU DDP

Gunakan kedua GPU dengan PyTorch DistributedDataParallel. `PRETRAIN_BATCH=32` adalah batch **per GPU**; dengan 2 GPU dan accumulation 1, effective global batch tetap 64 (sama dengan konfigurasi single-GPU 32×2). Jalankan `torchrun` agar kedua proses terikat ke GPU masing-masing.


In [ ]:
import os

CONFIG = {
    "PRETRAIN_BATCH": "32",       # per GPU
    "PRETRAIN_ACCUM": "1",       # global batch = 32 x 2 GPU x 1 = 64
    "PRETRAIN_STEPS": "6000",
    "PRETRAIN_WARMUP": "150",
    "PRETRAIN_PLATEAU_PATIENCE": "2",
    "PRETRAIN_PLATEAU_FACTOR": "0.5",
    "PRETRAIN_MIN_LR_SCALE": "0.125",
    "PRETRAIN_EARLY_STOP_PATIENCE": "8", # stop if validation stalls
    "PRETRAIN_EARLY_STOP_MIN_DELTA": "0.0001",
    "PRETRAIN_CKPT_EVERY": "500",
    "PRETRAIN_EVAL_EVERY": "500",
    "PRETRAIN_WORKERS": "2",      # per GPU process
    "PRETRAIN_PREFETCH": "2",
    "PRETRAIN_RESUME": "1",
    "MFM_TORCH_COMPILE": "0"
}

os.environ.update(CONFIG)

# Assert exactly two Kaggle GPUs before launching DDP.
import torch
assert torch.cuda.device_count() >= 2, f"Need 2 GPUs, found {torch.cuda.device_count()}"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(2)])

!torchrun --standalone --nnodes=1 --nproc_per_node=2 stage3_pretrain.py

sync_output("ckpt")


## 6. Verifikasi checkpoint

Pemeriksaan checkpoint/tokenizer sebelum fine-tuning.

In [ ]:
!python -u verify_ckpt.py 20

## 7. Fine-tuning multi-task

Effective batch tetap 256 (micro-batch 32 × accumulation 8). Checkpoint dipilih hanya berdasarkan XAUUSD 2021. Tahun 2022 dan 2023 tidak digunakan untuk pemilihan checkpoint.

In [ ]:
import os
os.environ['ST4_EPOCHS'] = '3'
os.environ['ST4_BATCH'] = '32'
os.environ['ST4_ACCUM'] = '8'
os.environ['ST4_LR'] = '1e-4'
os.environ['ST4_WARMUP_RATIO'] = '0.08'
os.environ['ST4_MIN_LR_RATIO'] = '0.10'
os.environ['ST4_PATIENCE'] = '3'
os.environ['ST4_STRIDE'] = '32'
os.environ['ST4_VAL_STRIDE'] = '4'
os.environ['ST4_TEST_STRIDE'] = '4'
os.environ['ST4_HELD_STRIDE'] = '8'
os.environ['ST4_AMP'] = '1'
os.environ['ST4_WORKERS'] = '2'
os.environ['ST4_PREFETCH'] = '2'
os.environ['MFM_TORCH_COMPILE'] = '0'
!python -u stage4_finetune.py
sync_output('ftune')

## 8. Kalibrasi dan evaluasi final

Kalibrasi dilakukan pada 2022 dan final test 2023. Jangan mengubah parameter atau memilih model berdasarkan hasil final test.

In [ ]:
!python -u validate_stage4.py
!python -u backtest_stage4.py
for folder in ['ftune', 'mkt', 'corpus', 'npz', 'ckpt']:
    sync_output(folder)

## 9. Artefak dan melanjutkan sesi

Artefak tersimpan di `/kaggle/working/mfm_output/`:
- `corpus/`: data mentah
- `npz/`: data hasil parsing
- `mkt/`: tokenizer, token, dan label
- `ckpt/`: checkpoint dan resume state
- `ftune/`: hasil fine-tuning, kalibrasi, dan backtest

Setelah proses selesai atau sebelum sesi berakhir, pilih **Save Version → Save & Run All** agar output ikut tersimpan. Untuk sesi berikutnya, buat/unggah Kaggle Dataset dari output tersebut, attach sebagai input, lalu isi `INPUT_DATASET_DIR`. Kaggle tidak menjamin filesystem `/kaggle/working` tetap tersedia di sesi baru.

Catat commit SHA, GPU, versi CUDA/PyTorch, konfigurasi, kelengkapan data, dan asumsi spread/komisi/slippage bersama hasil eksperimen. PPL = exp(cross-entropy), sehingga batas matematisnya 1, bukan 0.